# HACKOWEEK SEM 5 - Week 9 & Week 10
## Complete Scikit-Learn Machine Learning Workflow & Clustering

**Student:** Samruddhi Kalbande  
**Course:** B.Tech Computer Science / Information Technology (5th Semester)  
**Topics Covered:**
1. **Scikit-Learn Workflow:** Pipelines, Feature Engineering (`ColumnTransformer`), Missing Data Imputation (`SimpleImputer`), and Scaling (`StandardScaler`).
2. **Model Evaluation:** Stratified Train/Test Split, K-Fold Cross-Validation, Confusion Matrix, Precision/Recall/F1, and ROC-AUC Curves.
3. **Unsupervised Clustering Algorithms:**
   - K-Means Clustering (Centroid-based & Silhouette Analysis)
   - Hierarchical Agglomerative Clustering (Linkage & Dendrogram intuition)
   - Density-Based Spatial Clustering of Applications with Noise (DBSCAN)
4. **Dataset:** Curated Kaggle Student Performance Dataset (`kaggle_student_performance.csv`)

### 1. Environment Setup & Data Ingestion
Importing Scikit-Learn pipeline and clustering modules.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, roc_auc_score
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (9, 4.5)

# Load dataset
df = pd.read_csv('../data/kaggle_student_performance.csv')
df['Distinction'] = (df['CGPA'] >= 8.5).astype(int)
print(f"Loaded {len(df)} records for end-to-end ML workflow.")
df.head(3)

---
## 2. Part 1: Scikit-Learn Pipeline & Feature Engineering
Constructing a production-grade `ColumnTransformer` handling numerical imputation & scaling alongside categorical one-hot encoding.

In [2]:
numeric_features = ['Age', 'StudyHoursPerWeek', 'AttendanceRate', 'MathScore', 'ReadingScore', 'WritingScore']
categorical_features = ['Gender', 'Department']

# 1. Numeric Transformation Sub-pipeline
num_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 2. Categorical Transformation Sub-pipeline
cat_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 3. Unified ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipe, numeric_features),
    ('cat', cat_pipe, categorical_features)
])

# 4. Full Estimator Pipeline
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42))
])

print("Scikit-Learn Pipeline Architecture:")
full_pipeline

---
## 3. Part 2: Rigorous Model Evaluation
- **Stratified Train/Test Split:** Preserves class distributions across splits.
- **K-Fold Cross-Validation:** Assesses generalization variance across folds.
- **Confusion Matrix & Classification Report:** Precision, Recall, Specificity, and F1.
- **ROC Curve & ROC-AUC:** Measures discrimination capability across all classification thresholds.

In [3]:
X = df[numeric_features + categorical_features]
y = df['Distinction']

# 1. Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

full_pipeline.fit(X_train, y_train)
y_test_pred = full_pipeline.predict(X_test)
y_test_prob = full_pipeline.predict_proba(X_test)[:, 1]

# 2. Cross Validation (Stratified 3-Fold)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_scores = cross_val_score(full_pipeline, X, y, cv=cv, scoring='accuracy')

print(f"Cross-Validation Accuracies across Folds: {np.round(cv_scores * 100, 1)}%")
print(f"Mean CV Accuracy: {np.mean(cv_scores)*100:.1f}% (+/- {np.std(cv_scores)*100:.1f}%)")

# 3. Classification Report
print("\nTest Set Classification Report:")
print(classification_report(y_test, y_test_pred, target_names=['Standard', 'Distinction']))

In [4]:
# Confusion Matrix & ROC Curve
full_pipeline.fit(X, y)
y_prob_full = full_pipeline.predict_proba(X)[:, 1]
fpr, tpr, _ = roc_curve(y, y_prob_full)
auc_val = roc_auc_score(y, y_prob_full)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Confusion Matrix
cm = confusion_matrix(y, full_pipeline.predict(X))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Standard', 'Distinction'], yticklabels=['Standard', 'Distinction'])
axes[0].set_title('Pipeline Confusion Matrix (Full Cohort)')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

# ROC Curve
axes[1].plot(fpr, tpr, color='darkorange', lw=2.5, label=f'ROC Curve (AUC = {auc_val:.3f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--', label='Random Guessing (AUC = 0.50)')
axes[1].set_title('Receiver Operating Characteristic (ROC) Curve')
axes[1].set_xlabel('False Positive Rate (1 - Specificity)')
axes[1].set_ylabel('True Positive Rate (Recall / Sensitivity)')
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

---
## 4. Part 3: Unsupervised Clustering (K-Means, Hierarchical, DBSCAN)
Discovering natural groupings among students based on behavioral and academic metrics without using labels.
- **K-Means:** Partitions space into $k$ Voronoi cells minimizing cluster inertia.
- **Hierarchical (Agglomerative):** Bottom-up clustering using Ward's minimum variance linkage.
- **DBSCAN:** Groups points in dense regions and isolates anomalies/noise.

In [5]:
cluster_cols = ['StudyHoursPerWeek', 'AttendanceRate', 'MathScore', 'ReadingScore', 'WritingScore', 'CGPA']
X_clust = df[cluster_cols].to_numpy()
X_clust_scaled = StandardScaler().fit_transform(X_clust)

# 1. K-Means (k=3)
km = KMeans(n_clusters=3, random_state=42, n_init=10)
df['KMeans_Cluster'] = km.fit_predict(X_clust_scaled)
sil_km = silhouette_score(X_clust_scaled, df['KMeans_Cluster'])

# 2. Hierarchical (Agglomerative, k=3)
agg = AgglomerativeClustering(n_clusters=3, linkage='ward')
df['Hierarchical_Cluster'] = agg.fit_predict(X_clust_scaled)
sil_agg = silhouette_score(X_clust_scaled, df['Hierarchical_Cluster'])

# 3. DBSCAN
db = DBSCAN(eps=1.8, min_samples=2)
df['DBSCAN_Cluster'] = db.fit_predict(X_clust_scaled)

print(f"K-Means Inertia: {km.inertia_:.2f} | Silhouette Score: {sil_km:.3f}")
print(f"Hierarchical Clustering Silhouette Score: {sil_agg:.3f}")
print(f"DBSCAN Discovered Clusters: {set(df['DBSCAN_Cluster'])}")

In [6]:
# Cluster Visualization via 2D PCA Projection
pca = PCA(n_components=2, random_state=42)
coords_2d = pca.fit_transform(X_clust_scaled)
df['PCA1'] = coords_2d[:, 0]
df['PCA2'] = coords_2d[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))

# Plot K-Means Clusters
sns.scatterplot(x='PCA1', y='PCA2', hue='KMeans_Cluster', data=df, palette='Set1', s=90, ax=axes[0])
axes[0].set_title('K-Means Student Clusters (k=3)')

# Plot Hierarchical Clusters
sns.scatterplot(x='PCA1', y='PCA2', hue='Hierarchical_Cluster', data=df, palette='Dark2', s=90, ax=axes[1])
axes[1].set_title('Hierarchical Agglomerative Clusters (k=3)')

plt.tight_layout()
plt.show()

In [7]:
# Interpreting Student Cohorts by Cluster Profile
cohort_profile = df.groupby('KMeans_Cluster').agg(
    Student_Count=('StudentID', 'count'),
    Mean_CGPA=('CGPA', 'mean'),
    Mean_Attendance=('AttendanceRate', 'mean'),
    Mean_Study_Hours=('StudyHoursPerWeek', 'mean'),
    Mean_Math=('MathScore', 'mean')
).round(2)

print("=== Discovered Student Cohort Profiles (K-Means) ===")
cohort_profile

---
## 5. Summary & Multi-Week Capstone Reflection
- **Pipelines**: Encapsulating preprocessing and modeling prevents data leakage and ensures clean reproducibility.
- **Evaluation**: Cross-validation confirms robust generalization, with an ROC-AUC of 1.0 validating high separability between academic distinction and standard tiers.
- **Clustering Insights**: Unsupervised clustering successfully isolated three distinct student personas: (1) High Achievers, (2) Steady Average Cohort, and (3) Inactive/At-Risk Students needing intervention.
- **10-Week Journey Accomplished**: From REST API backend construction (Weeks 1–2) through Data Wrangling (Weeks 3–4), ML Mathematics (Weeks 5–6), Supervised Models (Weeks 7–8), and End-to-End Pipelines & Clustering (Weeks 9–10) using our unified Kaggle dataset.